# Phase 1 V2 — Notebook 03: FFIEC Data Extraction and Harmonisation

**Purpose.** Extract bank identifiers, institution metadata and the locked 20-concept FFIEC source set; prefer consolidated RCFD values and use equivalent RCON values as fallback; apply validated historical code bridges and Schedule RC-N delinquency formulas; standardise dates and missing values; validate units and signs; audit IDRSSD–quarter and CERT–quarter uniqueness; and save the harmonised V2 FFIEC panel.

**Scope boundary.** RIAD income-statement fields remain year-to-date and are labelled `_ytd`. This notebook does **not** construct quarterly flows, ratios, targets, lags or model predictors. Quarterly-flow conversion occurs in Notebook 05.

**Safety.** All outputs are written only under `Data Final/Version 2 Main Analysis`. Version 1 remains read only.


## 1. Mount Google Drive and import packages


In [ ]:
# Mount Google Drive and import packages

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import ast
import csv
import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys

import numpy as np
import pandas as pd

# Colab normally includes a Parquet engine. Install pyarrow only when
# the runtime image does not already provide one.
try:
    import pyarrow  # noqa: F401
except ImportError:
    print("Installing the missing pyarrow Parquet dependency...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyarrow"],
        check=True,
    )
    import pyarrow  # noqa: F401

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
print("Google Drive mounted successfully.")


## 2. Import V2 configuration and verify Notebook 02


In [ ]:
# Import V2 configuration and verify Notebook 02

PROJECT_ROOT = Path("/content/drive/MyDrive/Data Final")
CONFIG_PATH = PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"V2 config.py not found: {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
v2_config = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = v2_config
spec.loader.exec_module(v2_config)

VERSION_2_MAIN_TIMING_POLICY = v2_config.VERSION_2_MAIN_TIMING_POLICY
EXPECTED_POLICY_KEYS = v2_config.EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS
FFIEC_RAW_DIR = v2_config.FFIEC_RAW_DIR
V2_ROOT = v2_config.V2_ROOT
V2_OUTPUT_ROOT = v2_config.V2_OUTPUT_ROOT
V2_PROCESSED_DATA_DIR = v2_config.V2_PROCESSED_DATA_DIR
DATA_SCOPE = v2_config.DATA_SCOPE

NB02_OUTPUT_DIR = V2_OUTPUT_ROOT / "02_FFIEC_Variable_Inventory"
NB02_AUDIT_PATH = NB02_OUTPUT_DIR / "audit_report_v2.json"
NB02_VARIABLE_QUARTER_PATH = NB02_OUTPUT_DIR / "ffiec_variable_quarter_availability_v2.csv"
NB03_OUTPUT_DIR = V2_OUTPUT_ROOT / "03_FFIEC_Data_Extraction_and_Harmonisation"
HARMONISED_PANEL_PATH = V2_PROCESSED_DATA_DIR / "harmonised_ffiec_panel_v2.parquet"

assert tuple(VERSION_2_MAIN_TIMING_POLICY) == EXPECTED_POLICY_KEYS
assert VERSION_2_MAIN_TIMING_POLICY["analysis_version"] == "v2_bank_lag1_main"
assert VERSION_2_MAIN_TIMING_POLICY["archived_v1_read_only"] is True
assert NB03_OUTPUT_DIR.is_relative_to(V2_ROOT)
assert HARMONISED_PANEL_PATH.is_relative_to(V2_ROOT)

required_paths = (
    FFIEC_RAW_DIR,
    NB02_AUDIT_PATH,
    NB02_VARIABLE_QUARTER_PATH,
)
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required Notebook 02 files or FFIEC folder are missing:\n"
        + "\n".join(missing_paths)
    )

with NB02_AUDIT_PATH.open("r", encoding="utf-8") as file:
    nb02_audit = json.load(file)
if nb02_audit.get("overall_status") != "PASS":
    raise ValueError("Notebook 02 V2 did not finish with PASS status.")
if nb02_audit.get("missing_core_concepts") not in ([], None):
    raise ValueError(f"Notebook 02 has missing core concepts: {nb02_audit['missing_core_concepts']}")
if nb02_audit.get("missing_reporting_quarters") not in ([], None):
    raise ValueError("Notebook 02 did not confirm complete reporting-quarter coverage.")

NB03_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
V2_PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

variable_quarter_availability = pd.read_csv(
    NB02_VARIABLE_QUARTER_PATH,
    dtype={"variable_code": "string", "quarter": "string"},
    low_memory=False,
)
variable_quarter_availability["variable_code"] = (
    variable_quarter_availability["variable_code"].str.upper().str.strip()
)
variable_quarter_availability["year"] = pd.to_numeric(
    variable_quarter_availability["year"], errors="coerce"
).astype("Int64")

print("Notebook 02 V2 status: PASS")
print(f"Analysis version: {VERSION_2_MAIN_TIMING_POLICY['analysis_version']}")
print(f"FFIEC raw folder: {FFIEC_RAW_DIR}")
print(f"Notebook 03 output: {NB03_OUTPUT_DIR}")
print(f"Harmonised panel: {HARMONISED_PANEL_PATH}")


## 3. Define and lock the validated 20-concept source map

This map reuses the previously validated Version 1 raw-code decisions so the V1–V2 comparison is not confounded by changing source definitions. V2 changes predictor timing later; it does not redefine the underlying FFIEC concepts.


In [ ]:
# Define and lock the validated 20-concept source map

source_mapping_records = [
    ("accumulated_other_comprehensive_income", "quarter_end_stock", "RCFDB530", "RCONB530", "Prefer RCFD; fallback to equivalent RCON.", "Negative values are valid and retained."),
    ("allowance_credit_losses", "quarter_end_stock", "RCFD3123", "RCON3123", "Prefer RCFD; fallback to equivalent RCON.", "Negative exceptions are flagged; values are not clipped."),
    ("available_for_sale_securities", "quarter_end_stock", "RCFD1773", "RCON1773", "Prefer RCFD; fallback to equivalent RCON.", "Expected non-negative; exceptions are flagged."),
    ("fed_funds_sold", "quarter_end_stock", "2001: RCFD1350/RCON1350", "2002+: RCONB987 + RCFDB989/RCONB989", "Apply the validated date bridge; prefer RCFD for equivalent pair B989.", "Expected non-negative; missing components are not automatically converted to zero."),
    ("held_to_maturity_securities", "quarter_end_stock", "2001-2018: RCFD1754", "RCON1754; 2019+: RCFDJJ34/RCONJJ34", "Apply the validated date bridge and prefer RCFD within each equivalent pair.", "Expected non-negative; exceptions are flagged."),
    ("net_income", "year_to_date_flow", "RIAD4340", "", "Direct RIAD YTD item; quarterly conversion deferred to Notebook 05.", "Genuine negative income is retained."),
    ("net_interest_income", "year_to_date_flow", "RIAD4074", "", "Direct RIAD YTD item; quarterly conversion deferred to Notebook 05.", "Genuine negative values are retained."),
    ("nonaccrual_loans", "quarter_end_stock", "2001-2016 historical Schedule RC-N formula", "2017+: RCFD1403/RCON1403", "Apply the locked form- and era-specific component registry.", "Expected non-negative; incomplete component sets remain missing."),
    ("other_real_estate_owned", "quarter_end_stock", "RCFD2150", "RCON2150", "Prefer RCFD; fallback to equivalent RCON.", "Expected non-negative; exceptions are flagged."),
    ("past_due_90_days", "quarter_end_stock", "2001-2016 historical Schedule RC-N formula", "2017+: RCFD1407/RCON1407", "Apply the locked form- and era-specific component registry.", "Expected non-negative; incomplete component sets remain missing."),
    ("provision_credit_losses", "year_to_date_flow", "2001-2018: RIAD4230", "2019+: RIADJJ33", "Apply the validated date bridge; quarterly conversion deferred to Notebook 05.", "Genuine negative provisions or recoveries are retained."),
    ("total_assets", "quarter_end_stock", "RCFD2170", "RCON2170", "Prefer consolidated RCFD; fallback to equivalent RCON.", "Must be positive; irrecoverable non-positive or missing rows are excluded and audited."),
    ("total_deposits", "quarter_end_stock", "RCON2200", "RCFN2200", "Domestic deposits plus foreign-office deposits; unavailable foreign-office amount is zero only when domestic deposits are present.", "Expected non-negative; exceptions are flagged."),
    ("total_equity_capital", "quarter_end_stock", "RCFD3210", "RCON3210", "Prefer RCFD; fallback to equivalent RCON.", "Negative equity is economically meaningful and retained."),
    ("total_interest_expense", "year_to_date_flow", "RIAD4073", "", "Direct RIAD YTD item; quarterly conversion deferred to Notebook 05.", "Reported sign is retained."),
    ("total_interest_income", "year_to_date_flow", "RIAD4107", "", "Direct RIAD YTD item; quarterly conversion deferred to Notebook 05.", "Reported sign is retained."),
    ("total_loans", "quarter_end_stock", "RCFDB528", "RCONB528", "Prefer RCFD; fallback to equivalent RCON.", "Expected non-negative; exceptions are flagged."),
    ("total_noninterest_expense", "year_to_date_flow", "RIAD4093", "", "Direct RIAD YTD item; quarterly conversion deferred to Notebook 05.", "Reported sign is retained."),
    ("total_noninterest_income", "year_to_date_flow", "RIAD4079", "", "Direct RIAD YTD item; quarterly conversion deferred to Notebook 05.", "Genuine negative values are retained."),
    ("trading_assets", "quarter_end_stock", "RCFD3545", "RCON3545", "Prefer RCFD; fallback to equivalent RCON.", "Expected non-negative; exceptions are flagged."),
]
final_source_mapping = pd.DataFrame(
    source_mapping_records,
    columns=[
        "concept", "measurement_basis", "primary_source", "fallback_or_bridge_source",
        "mapping_method", "sign_policy",
    ],
)
final_source_mapping["unit"] = "USD thousands"
final_source_mapping["mapping_origin"] = "locked_version_1_source_mapping_reused_for_v2"

expected_concepts = {
    "accumulated_other_comprehensive_income", "allowance_credit_losses",
    "available_for_sale_securities", "fed_funds_sold", "held_to_maturity_securities",
    "net_income", "net_interest_income", "nonaccrual_loans", "other_real_estate_owned",
    "past_due_90_days", "provision_credit_losses", "total_assets", "total_deposits",
    "total_equity_capital", "total_interest_expense", "total_interest_income",
    "total_loans", "total_noninterest_expense", "total_noninterest_income", "trading_assets",
}
assert len(final_source_mapping) == 20
assert final_source_mapping["concept"].is_unique
assert set(final_source_mapping["concept"]) == expected_concepts
assert not final_source_mapping.isna().any().any()

print("Validated 20-concept source map defined.")
display(final_source_mapping)


## 4. Define the locked historical Schedule RC-N formula registry


In [ ]:
# Define the locked historical Schedule RC-N formula registry

formula_specs = []

def add_formula(concept, form, start_year, end_year, component_codes, formula_method):
    formula_specs.append(
        {
            "formula_id": f"{concept}__{form}__{start_year}_{end_year}",
            "concept": concept,
            "form": form,
            "start_year": start_year,
            "end_year": end_year,
            "component_codes": component_codes,
            "component_count": len(component_codes),
            "formula_method": formula_method,
        }
    )

# Past due 90+ days — FFIEC 031.
pd31_2001 = ["RCON2769", "RCON3494", "RCON5399", "RCON5402", "RCON3500", "RCON3503", "RCFNB573", "RCFD1597", "RCFDB576", "RCFDB579", "RCFD5378", "RCFD5381", "RCFD1252", "RCFD1255", "RCFD5390", "RCFD5460", "RCFD1258", "RCFD1272"]
pd31_2002_2006 = ["RCON2769", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCON3503", "RCFNB573", "RCFD1597", "RCFDB576", "RCFDB579", "RCFD5378", "RCFD5381", "RCFD1252", "RCFD1255", "RCFD5390", "RCFD5460", "RCFD1258", "RCFD1272"]
pd31_2007 = ["RCON2769", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCON3503", "RCFNB573", "RCFD1597", "RCFDB576", "RCFDB579", "RCFD5378", "RCFD5381", "RCFD1252", "RCFD1255", "RCFD5390", "RCFD5460", "RCFDF167", "RCFDF170"]
pd31_2008_2010 = ["RCONF174", "RCONF175", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCONF180", "RCONF181", "RCFNB573", "RCFD1597", "RCFDB576", "RCFDB579", "RCFD5378", "RCFD5381", "RCFD1252", "RCFD1255", "RCFD5390", "RCFD5460", "RCFDF167", "RCFDF170"]
pd31_2011_2016 = ["RCONF174", "RCONF175", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCONF180", "RCONF181", "RCFNB573", "RCFD1597", "RCFDB576", "RCFDK214", "RCFDK217", "RCFD5378", "RCFD5381", "RCFD1252", "RCFD1255", "RCFD5390", "RCFD5460", "RCFDF167", "RCFDF170"]
add_formula("past_due_90_days", "FFIEC_031", 2001, 2001, pd31_2001, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_031", 2002, 2006, pd31_2002_2006, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_031", 2007, 2007, pd31_2007, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_031", 2008, 2010, pd31_2008_2010, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_031", 2011, 2016, pd31_2011_2016, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_031", 2017, 2026, ["RCFD1407"], "Use direct total RC-N item 9, column B.")

# Past due 90+ days — FFIEC 041/051.
pd41_2001 = ["RCON2769", "RCON3494", "RCON5399", "RCON5402", "RCON3500", "RCON3503", "RCONB835", "RCONB576", "RCONB579", "RCON5390", "RCON5460", "RCON1227", "RCON1607"]
pd41_2002_2007 = ["RCON2769", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCON3503", "RCONB835", "RCONB576", "RCONB579", "RCON5390", "RCON5460", "RCON1227", "RCON1607"]
pd41_2008_2010 = ["RCONF174", "RCONF175", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCONF180", "RCONF181", "RCONB835", "RCONB576", "RCONB579", "RCON5390", "RCON5460", "RCON1227", "RCON1607"]
pd41_2011_2016 = ["RCONF174", "RCONF175", "RCON3494", "RCON5399", "RCONC237", "RCONC239", "RCON3500", "RCONF180", "RCONF181", "RCONB835", "RCONB576", "RCONK214", "RCONK217", "RCON5390", "RCON5460", "RCON1227", "RCON1607"]
add_formula("past_due_90_days", "FFIEC_041_051", 2001, 2001, pd41_2001, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_041_051", 2002, 2007, pd41_2002_2007, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_041_051", 2008, 2010, pd41_2008_2010, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_041_051", 2011, 2016, pd41_2011_2016, "Sum validated Schedule RC-N column B components.")
add_formula("past_due_90_days", "FFIEC_041_051", 2017, 2026, ["RCON1407"], "Use direct total RC-N item 9, column B.")

# Nonaccrual real-estate components.
na_re31_2001 = ["RCON3492", "RCON3495", "RCON5400", "RCON5403", "RCON3501", "RCON3504", "RCFNB574"]
na_re31_2002_2007 = ["RCON3492", "RCON3495", "RCON5400", "RCONC229", "RCONC230", "RCON3501", "RCON3504", "RCFNB574"]
na_re31_2008_2016 = ["RCONF176", "RCONF177", "RCON3495", "RCON5400", "RCONC229", "RCONC230", "RCON3501", "RCONF182", "RCONF183", "RCFNB574"]
na_re41_2001 = ["RCON3492", "RCON3495", "RCON5400", "RCON5403", "RCON3501", "RCON3504"]
na_re41_2002_2007 = ["RCON3492", "RCON3495", "RCON5400", "RCONC229", "RCONC230", "RCON3501", "RCON3504"]
na_re41_2008_2016 = ["RCONF176", "RCONF177", "RCON3495", "RCON5400", "RCONC229", "RCONC230", "RCON3501", "RCONF182", "RCONF183"]

# Nonaccrual remaining-loan components.
na_other31_2001_2006 = ["RCFDB577", "RCFDB580", "RCFD5391", "RCFD1253", "RCFD1256", "RCFD1583", "RCFD1259", "RCFD1791", "RCFD5379", "RCFD5382", "RCFD5461"]
na_other31_2007_2010 = ["RCFDB577", "RCFDB580", "RCFD5391", "RCFD1253", "RCFD1256", "RCFD1583", "RCFDF168", "RCFDF171", "RCFD5379", "RCFD5382", "RCFD5461"]
na_other31_2011_2016 = ["RCFDB577", "RCFDK215", "RCFDK218", "RCFD5391", "RCFD1253", "RCFD1256", "RCFD1583", "RCFDF168", "RCFDF171", "RCFD5379", "RCFD5382", "RCFD5461"]
na_other41_2001_2010 = ["RCON1228", "RCON1608", "RCONB836", "RCONB577", "RCONB580", "RCON5391", "RCON5461"]
na_other41_2011_2016 = ["RCON1228", "RCON1608", "RCONB836", "RCONB577", "RCONK215", "RCONK218", "RCON5391", "RCON5461"]

add_formula("nonaccrual_loans", "FFIEC_031", 2001, 2001, na_re31_2001 + na_other31_2001_2006, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_031", 2002, 2006, na_re31_2002_2007 + na_other31_2001_2006, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_031", 2007, 2007, na_re31_2002_2007 + na_other31_2007_2010, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_031", 2008, 2010, na_re31_2008_2016 + na_other31_2007_2010, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_031", 2011, 2016, na_re31_2008_2016 + na_other31_2011_2016, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_031", 2017, 2026, ["RCFD1403"], "Use direct total RC-N item 9, column C.")
add_formula("nonaccrual_loans", "FFIEC_041_051", 2001, 2001, na_re41_2001 + na_other41_2001_2010, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_041_051", 2002, 2007, na_re41_2002_2007 + na_other41_2001_2010, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_041_051", 2008, 2010, na_re41_2008_2016 + na_other41_2001_2010, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_041_051", 2011, 2016, na_re41_2008_2016 + na_other41_2011_2016, "Sum validated Schedule RC-N column C components.")
add_formula("nonaccrual_loans", "FFIEC_041_051", 2017, 2026, ["RCON1403"], "Use direct total RC-N item 9, column C.")

final_historical_formula_registry = pd.DataFrame(formula_specs)
assert len(final_historical_formula_registry) == 22
assert final_historical_formula_registry["formula_id"].is_unique
assert final_historical_formula_registry.apply(
    lambda row: len(row["component_codes"]) == row["component_count"], axis=1
).all()
assert final_historical_formula_registry.apply(
    lambda row: len(row["component_codes"]) == len(set(row["component_codes"])), axis=1
).all()

formula_coverage_records = []
for formula in final_historical_formula_registry.itertuples(index=False):
    for year in range(int(formula.start_year), int(formula.end_year) + 1):
        formula_coverage_records.append(
            {"concept": formula.concept, "form": formula.form, "year": year, "formula_id": formula.formula_id}
        )
formula_coverage = pd.DataFrame(formula_coverage_records)
expected_formula_coverage = {
    (concept, form, year)
    for concept in ("past_due_90_days", "nonaccrual_loans")
    for form in ("FFIEC_031", "FFIEC_041_051")
    for year in range(2001, 2027)
}
observed_formula_coverage = set(
    formula_coverage[["concept", "form", "year"]].itertuples(index=False, name=None)
)
assert observed_formula_coverage == expected_formula_coverage
assert not formula_coverage.duplicated(["concept", "form", "year"]).any()

print("Locked 22-window Schedule RC-N registry defined and coverage verified.")
display(final_historical_formula_registry)


## 5. Validate every locked code against Notebook 02 historical availability


In [ ]:
# Validate every locked code against Notebook 02 historical availability

direct_code_requirements = [
    ("accumulated_other_comprehensive_income", "RCFDB530", 2001, 2026),
    ("accumulated_other_comprehensive_income", "RCONB530", 2001, 2026),
    ("allowance_credit_losses", "RCFD3123", 2001, 2026),
    ("allowance_credit_losses", "RCON3123", 2001, 2026),
    ("available_for_sale_securities", "RCFD1773", 2001, 2026),
    ("available_for_sale_securities", "RCON1773", 2001, 2026),
    ("fed_funds_sold", "RCFD1350", 2001, 2001),
    ("fed_funds_sold", "RCON1350", 2001, 2001),
    ("fed_funds_sold", "RCONB987", 2002, 2026),
    ("fed_funds_sold", "RCFDB989", 2002, 2026),
    ("fed_funds_sold", "RCONB989", 2002, 2026),
    ("held_to_maturity_securities", "RCFD1754", 2001, 2018),
    ("held_to_maturity_securities", "RCON1754", 2001, 2018),
    ("held_to_maturity_securities", "RCFDJJ34", 2019, 2026),
    ("held_to_maturity_securities", "RCONJJ34", 2019, 2026),
    ("net_income", "RIAD4340", 2001, 2026),
    ("net_interest_income", "RIAD4074", 2001, 2026),
    ("other_real_estate_owned", "RCFD2150", 2001, 2026),
    ("other_real_estate_owned", "RCON2150", 2001, 2026),
    ("provision_credit_losses", "RIAD4230", 2001, 2018),
    ("provision_credit_losses", "RIADJJ33", 2019, 2026),
    ("total_assets", "RCFD2170", 2001, 2026),
    ("total_assets", "RCON2170", 2001, 2026),
    ("total_deposits", "RCON2200", 2001, 2026),
    ("total_deposits", "RCFN2200", 2001, 2026),
    ("total_equity_capital", "RCFD3210", 2001, 2026),
    ("total_equity_capital", "RCON3210", 2001, 2026),
    ("total_interest_expense", "RIAD4073", 2001, 2026),
    ("total_interest_income", "RIAD4107", 2001, 2026),
    ("total_loans", "RCFDB528", 2001, 2026),
    ("total_loans", "RCONB528", 2001, 2026),
    ("total_noninterest_expense", "RIAD4093", 2001, 2026),
    ("total_noninterest_income", "RIAD4079", 2001, 2026),
    ("trading_assets", "RCFD3545", 2001, 2026),
    ("trading_assets", "RCON3545", 2001, 2026),
]
availability_years_by_code = (
    variable_quarter_availability.dropna(subset=["year"])
    .assign(year=lambda frame: frame["year"].astype(int))
    .groupby("variable_code")["year"].apply(set).to_dict()
)

code_requirement_records = []
for concept, variable_code, start_year, end_year in direct_code_requirements:
    required_years = set(range(start_year, end_year + 1))
    observed_years = availability_years_by_code.get(variable_code, set())
    missing_years = sorted(required_years - observed_years)
    code_requirement_records.append(
        {
            "requirement_type": "direct_or_bridge",
            "concept": concept,
            "formula_id": None,
            "variable_code": variable_code,
            "start_year": start_year,
            "end_year": end_year,
            "missing_required_years": ";".join(map(str, missing_years)),
            "required_window_complete": len(missing_years) == 0,
        }
    )
for formula in final_historical_formula_registry.itertuples(index=False):
    required_years = set(range(int(formula.start_year), int(formula.end_year) + 1))
    for variable_code in formula.component_codes:
        observed_years = availability_years_by_code.get(variable_code, set())
        missing_years = sorted(required_years - observed_years)
        code_requirement_records.append(
            {
                "requirement_type": "historical_formula_component",
                "concept": formula.concept,
                "formula_id": formula.formula_id,
                "variable_code": variable_code,
                "start_year": int(formula.start_year),
                "end_year": int(formula.end_year),
                "missing_required_years": ";".join(map(str, missing_years)),
                "required_window_complete": len(missing_years) == 0,
            }
        )

locked_code_coverage_validation = pd.DataFrame(code_requirement_records)
incomplete_code_windows = locked_code_coverage_validation.loc[
    ~locked_code_coverage_validation["required_window_complete"]
].copy().reset_index(drop=True)

print(f"Locked code-window checks: {len(locked_code_coverage_validation):,}")
print(f"Incomplete required code windows: {len(incomplete_code_windows):,}")
if not incomplete_code_windows.empty:
    display(incomplete_code_windows)
    raise AssertionError(
        "Locked source codes are missing in required historical windows. "
        "Review locked_code_coverage_validation_v2.csv before extraction."
    )

print("All locked direct, bridge and historical-formula code windows are present.")


## 6. Lock mapping fingerprints and discover extraction files


In [ ]:
# Lock mapping fingerprints and discover extraction files

def dataframe_sha256(dataframe):
    stable = dataframe.copy()
    for column in stable.columns:
        stable[column] = stable[column].map(
            lambda value: json.dumps(value, separators=(",", ":"))
            if isinstance(value, (list, tuple, dict)) else value
        )
    stable_csv = stable.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(stable_csv.encode("utf-8")).hexdigest()

mapping_sha256 = dataframe_sha256(final_source_mapping)
formula_registry_sha256 = dataframe_sha256(final_historical_formula_registry)
mapping_lock_manifest = {
    "mapping_version": "NB03_V2_LOCKED_SOURCE_MAP",
    "lock_status": "LOCKED",
    "concept_count": len(final_source_mapping),
    "historical_formula_windows": len(final_historical_formula_registry),
    "source_mapping_sha256": mapping_sha256,
    "formula_registry_sha256": formula_registry_sha256,
    "mapping_origin": "validated_version_1_mapping_reused_for_v2",
}

RAW_CODE_PATTERN = re.compile(r"\b(?:RCFD|RCON|RCFN|RIAD)[A-Z0-9]{4}\b")
required_raw_codes = set()
for row in direct_code_requirements:
    required_raw_codes.add(row[1])
for component_codes in final_historical_formula_registry["component_codes"]:
    required_raw_codes.update(component_codes)
required_raw_codes = sorted(required_raw_codes)

supported_extensions = {".txt", ".csv", ".tsv", ".dat"}
ffiec_files = sorted(
    path for path in FFIEC_RAW_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in supported_extensions
    and not any(part.startswith(".") for part in path.relative_to(FFIEC_RAW_DIR).parts)
)
if not ffiec_files:
    raise FileNotFoundError("No supported FFIEC text files were found.")

def detect_encoding(path):
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as file:
                file.readline()
            return encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Unable to detect encoding: {path}")

def detect_delimiter(path, encoding):
    with path.open("r", encoding=encoding, errors="replace") as file:
        first_line = file.readline()
    counts = {"\t": first_line.count("\t"), ",": first_line.count(","), "|": first_line.count("|"), ";": first_line.count(";")}
    delimiter = max(counts, key=counts.get)
    if counts[delimiter] == 0:
        raise ValueError(f"Delimiter not detected: {path}")
    return delimiter

def extract_source_year(path):
    years = re.findall(r"(?<!\d)(20\d{2})(?!\d)", str(path.relative_to(FFIEC_RAW_DIR)))
    valid = [int(year) for year in years if 2001 <= int(year) <= 2026]
    return valid[-1] if valid else None

extraction_file_records = []
eligible_raw_files = []
for path in ffiec_files:
    try:
        encoding = detect_encoding(path)
        delimiter = detect_delimiter(path, encoding)
        header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0).columns.tolist()
        header_lookup = {str(column).strip().upper(): column for column in header}
        source_year = extract_source_year(path)
        usable = (
            source_year is not None
            and "REPORTING PERIOD END DATE" in header_lookup
            and "IDRSSD" in header_lookup
        )
        extraction_file_records.append(
            {
                "relative_path": str(path.relative_to(FFIEC_RAW_DIR)),
                "source_year": source_year,
                "encoding": encoding,
                "delimiter": repr(delimiter),
                "header_columns": len(header),
                "required_codes_available": sum(code in header_lookup for code in required_raw_codes),
                "usable": usable,
                "error": None,
            }
        )
        if usable:
            eligible_raw_files.append(
                {"path": path, "year": source_year, "encoding": encoding, "delimiter": delimiter}
            )
    except Exception as error:
        extraction_file_records.append(
            {
                "relative_path": str(path.relative_to(FFIEC_RAW_DIR)),
                "source_year": extract_source_year(path),
                "encoding": None,
                "delimiter": None,
                "header_columns": np.nan,
                "required_codes_available": 0,
                "usable": False,
                "error": f"{type(error).__name__}: {error}",
            }
        )

raw_extraction_file_inventory = pd.DataFrame(extraction_file_records)
eligible_file_years = {record["year"] for record in eligible_raw_files}
expected_years = set(range(2001, 2027))
missing_file_years = sorted(expected_years - eligible_file_years)

print(f"Locked raw codes required: {len(required_raw_codes):,}")
print(f"Usable FFIEC source files: {len(eligible_raw_files):,}")
print(f"Missing source-file years: {missing_file_years}")
if missing_file_years:
    raise AssertionError(f"FFIEC extraction files are missing years: {missing_file_years}")


## 7. Read, standardise and merge annual FFIEC file parts


In [ ]:
# Read, standardise and merge annual FFIEC file parts

raw_metadata_columns = {
    "REPORTING PERIOD END DATE": "report_date",
    "IDRSSD": "idrssd",
    "FDIC CERTIFICATE NUMBER": "cert",
    "FINANCIAL INSTITUTION NAME": "bank_name",
    "FINANCIAL INSTITUTION STATE": "bank_state",
    "FINANCIAL INSTITUTION FILING TYPE": "filing_form",
}
missing_text_tokens = {"", " ", "NA", "N/A", "NULL", "NONE", "."}

def read_raw_file_part(record):
    file_path = record["path"]
    header = pd.read_csv(
        file_path, sep=record["delimiter"], encoding=record["encoding"], nrows=0
    ).columns.tolist()
    header_lookup = {str(column).strip().upper(): column for column in header}
    selected_original_columns = []
    for normalised_metadata in raw_metadata_columns:
        original = header_lookup.get(normalised_metadata)
        if original is not None:
            selected_original_columns.append(original)
    available_codes = [code for code in required_raw_codes if code in header_lookup]
    selected_original_columns.extend(header_lookup[code] for code in available_codes)
    selected_original_columns = list(dict.fromkeys(selected_original_columns))

    raw_part = pd.read_csv(
        file_path,
        sep=record["delimiter"],
        encoding=record["encoding"],
        header=0,
        skiprows=[1],
        usecols=selected_original_columns,
        dtype=str,
        low_memory=False,
        keep_default_na=False,
    )
    original_rows = len(raw_part)
    rename_dictionary = {}
    for column in raw_part.columns:
        normalised = str(column).strip().upper()
        if normalised in required_raw_codes:
            rename_dictionary[column] = normalised
        elif normalised in raw_metadata_columns:
            rename_dictionary[column] = raw_metadata_columns[normalised]
    raw_part = raw_part.rename(columns=rename_dictionary)

    if "report_date" not in raw_part or "idrssd" not in raw_part:
        raise ValueError(f"Required identifiers missing from {file_path}")
    parsed_dates = pd.to_datetime(raw_part["report_date"], errors="coerce")
    invalid_report_date_rows = int(parsed_dates.isna().sum())
    non_quarter_end_raw_rows = int((parsed_dates.notna() & ~parsed_dates.dt.is_quarter_end).sum())
    raw_part["report_date"] = parsed_dates.dt.to_period("Q").dt.end_time.dt.normalize()
    raw_part["idrssd"] = pd.to_numeric(raw_part["idrssd"], errors="coerce").astype("Int64")
    if "cert" in raw_part:
        raw_part["cert"] = pd.to_numeric(raw_part["cert"], errors="coerce").astype("Int64")
    else:
        raw_part["cert"] = pd.Series(pd.NA, index=raw_part.index, dtype="Int64")

    for column in ("bank_name", "bank_state", "filing_form"):
        if column not in raw_part:
            raw_part[column] = pd.Series(pd.NA, index=raw_part.index, dtype="string")
        raw_part[column] = raw_part[column].astype("string").str.strip().replace("", pd.NA)
    raw_part["filing_form"] = raw_part["filing_form"].str.extract(r"(031|041|051)", expand=False)

    numeric_conversion_failures = 0
    for variable_code in available_codes:
        raw_text = raw_part[variable_code].astype("string").str.strip()
        standard_missing = raw_text.str.upper().isin(missing_text_tokens)
        cleaned = (
            raw_text.mask(standard_missing)
            .str.replace(",", "", regex=False)
            .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        )
        numeric_value = pd.to_numeric(cleaned, errors="coerce")
        numeric_conversion_failures += int(cleaned.notna().sum() - numeric_value.notna().sum())
        raw_part[variable_code] = numeric_value

    invalid_idrssd_rows = int(raw_part["idrssd"].isna().sum())
    duplicate_key_rows = int(
        raw_part.duplicated(["idrssd", "report_date"], keep=False).sum()
    )
    file_audit = {
        "relative_path": str(file_path.relative_to(FFIEC_RAW_DIR)),
        "source_year": record["year"],
        "original_rows": original_rows,
        "invalid_report_date_rows": invalid_report_date_rows,
        "non_quarter_end_raw_rows": non_quarter_end_raw_rows,
        "valid_rows": len(raw_part),
        "available_required_codes": len(available_codes),
        "invalid_idrssd_rows": invalid_idrssd_rows,
        "duplicate_key_rows": duplicate_key_rows,
        "numeric_conversion_failures": numeric_conversion_failures,
    }
    return raw_part, file_audit

def merge_year_parts(year_parts, source_year):
    merge_keys = ["idrssd", "report_date"]
    merged = year_parts[0].copy()
    conflict_records = []
    for part_number, next_part in enumerate(year_parts[1:], start=2):
        common_columns = sorted(
            set(merged.columns).intersection(next_part.columns).difference(merge_keys)
        )
        merged = merged.merge(
            next_part,
            on=merge_keys,
            how="outer",
            suffixes=("", "__new"),
            validate="one_to_one",
        )
        for column in common_columns:
            new_column = column + "__new"
            both_present = merged[column].notna() & merged[new_column].notna()
            conflicts = int(
                (
                    merged.loc[both_present, column].astype(str)
                    != merged.loc[both_present, new_column].astype(str)
                ).sum()
            )
            conflict_records.append(
                {
                    "source_year": source_year,
                    "part_number": part_number,
                    "column": column,
                    "conflicting_values": conflicts,
                }
            )
            merged[column] = merged[column].combine_first(merged[new_column])
            merged = merged.drop(columns=[new_column])
    merged["source_year"] = source_year
    return merged, conflict_records

files_by_year = {year: [] for year in range(2001, 2027)}
for record in eligible_raw_files:
    files_by_year[record["year"]].append(record)

annual_panels = []
file_read_audit_records = []
merge_conflict_records = []
for source_year in tqdm(range(2001, 2027), desc="Extracting FFIEC years"):
    year_records = files_by_year[source_year]
    year_parts = []
    for record in sorted(year_records, key=lambda item: str(item["path"])):
        raw_part, file_audit = read_raw_file_part(record)
        file_read_audit_records.append(file_audit)
        if file_audit["invalid_report_date_rows"] > 0:
            raise AssertionError(f"Invalid report dates in {record['path']}")
        if file_audit["invalid_idrssd_rows"] > 0:
            raise AssertionError(f"Invalid IDRSSD rows in {record['path']}")
        if file_audit["duplicate_key_rows"] > 0:
            raise AssertionError(f"Duplicate IDRSSD-quarter rows inside {record['path']}")
        year_parts.append(raw_part)
    year_panel, conflict_records = merge_year_parts(year_parts, source_year)
    annual_panels.append(year_panel)
    merge_conflict_records.extend(conflict_records)

file_read_audit = pd.DataFrame(file_read_audit_records)
part_merge_conflicts = pd.DataFrame(
    merge_conflict_records,
    columns=["source_year", "part_number", "column", "conflicting_values"],
)
raw_required_panel = pd.concat(annual_panels, ignore_index=True, sort=False)
for variable_code in required_raw_codes:
    if variable_code not in raw_required_panel:
        raw_required_panel[variable_code] = np.nan

raw_required_panel["filing_form_group"] = np.select(
    [
        raw_required_panel["filing_form"].eq("031"),
        raw_required_panel["filing_form"].isin(["041", "051"]),
    ],
    ["FFIEC_031", "FFIEC_041_051"],
    default="UNSUPPORTED",
)
raw_required_panel = raw_required_panel.sort_values(
    ["report_date", "idrssd"]
).reset_index(drop=True)

merge_value_conflicts = int(part_merge_conflicts["conflicting_values"].sum()) if not part_merge_conflicts.empty else 0
numeric_conversion_failures = int(file_read_audit["numeric_conversion_failures"].sum())
unsupported_form_rows = int(raw_required_panel["filing_form_group"].eq("UNSUPPORTED").sum())
non_quarter_end_rows = int((~raw_required_panel["report_date"].dt.is_quarter_end).sum())

print(f"Raw harmonisation rows: {len(raw_required_panel):,}")
print(f"Merge-value conflicts: {merge_value_conflicts:,}")
print(f"Numeric conversion failures: {numeric_conversion_failures:,}")
print(f"Unsupported filing-form rows: {unsupported_form_rows:,}")


## 8. Construct the 20 harmonised concepts without imputing or clipping


In [ ]:
# Construct the 20 harmonised concepts without imputing or clipping

raw_panel = raw_required_panel
report_year = raw_panel["report_date"].dt.year
form_group = raw_panel["filing_form_group"]

def prefer_rcfd(rcfd_code, rcon_code):
    rcfd = pd.to_numeric(raw_panel[rcfd_code], errors="coerce")
    rcon = pd.to_numeric(raw_panel[rcon_code], errors="coerce")
    value = rcfd.combine_first(rcon)
    source = pd.Series(pd.NA, index=raw_panel.index, dtype="string")
    source.loc[rcfd.notna()] = rcfd_code
    source.loc[rcfd.isna() & rcon.notna()] = rcon_code
    return value, source

def construct_historical_formula(concept_name):
    result = pd.Series(np.nan, index=raw_panel.index, dtype="float64")
    source = pd.Series(pd.NA, index=raw_panel.index, dtype="string")
    assignment_count = pd.Series(0, index=raw_panel.index, dtype="int64")
    audit_records = []
    formulas = final_historical_formula_registry.loc[
        final_historical_formula_registry["concept"].eq(concept_name)
    ]
    for formula in formulas.itertuples(index=False):
        formula_mask = (
            report_year.between(int(formula.start_year), int(formula.end_year))
            & form_group.eq(formula.form)
        )
        component_values = raw_panel.loc[formula_mask, formula.component_codes]
        calculated = component_values.sum(axis=1, min_count=len(formula.component_codes))
        result.loc[formula_mask] = calculated
        source.loc[formula_mask] = formula.formula_id
        assignment_count.loc[formula_mask] += 1
        audit_records.append(
            {
                "formula_id": formula.formula_id,
                "concept": concept_name,
                "form": formula.form,
                "start_year": int(formula.start_year),
                "end_year": int(formula.end_year),
                "component_count": len(formula.component_codes),
                "assigned_rows": int(formula_mask.sum()),
                "complete_input_rows": int(component_values.notna().all(axis=1).sum()),
                "incomplete_input_rows": int((~component_values.notna().all(axis=1)).sum()),
                "nonmissing_output_rows": int(calculated.notna().sum()),
                "missing_output_rows": int(calculated.isna().sum()),
            }
        )
    return result, source, assignment_count, audit_records

metadata_columns = [
    "report_date", "idrssd", "cert", "bank_name", "bank_state",
    "filing_form", "filing_form_group", "source_year",
]
concept_panel = raw_panel[metadata_columns].copy()

direct_pairs = {
    "accumulated_other_comprehensive_income": ("RCFDB530", "RCONB530"),
    "allowance_credit_losses": ("RCFD3123", "RCON3123"),
    "available_for_sale_securities": ("RCFD1773", "RCON1773"),
    "other_real_estate_owned": ("RCFD2150", "RCON2150"),
    "total_assets": ("RCFD2170", "RCON2170"),
    "total_equity_capital": ("RCFD3210", "RCON3210"),
    "total_loans": ("RCFDB528", "RCONB528"),
    "trading_assets": ("RCFD3545", "RCON3545"),
}
for concept, (rcfd_code, rcon_code) in direct_pairs.items():
    concept_panel[concept], concept_panel[f"{concept}__source"] = prefer_rcfd(rcfd_code, rcon_code)

fed_2001, fed_2001_source = prefer_rcfd("RCFD1350", "RCON1350")
resale, resale_source = prefer_rcfd("RCFDB989", "RCONB989")
fed_2002 = pd.concat([raw_panel["RCONB987"], resale], axis=1).sum(axis=1, min_count=1)
concept_panel["fed_funds_sold"] = fed_2002
concept_panel["fed_funds_sold__source"] = "RCONB987 + preferred RCFDB989/RCONB989"
mask_2001 = report_year.eq(2001)
concept_panel.loc[mask_2001, "fed_funds_sold"] = fed_2001.loc[mask_2001]
concept_panel.loc[mask_2001, "fed_funds_sold__source"] = fed_2001_source.loc[mask_2001]

htm_legacy, htm_legacy_source = prefer_rcfd("RCFD1754", "RCON1754")
htm_new, htm_new_source = prefer_rcfd("RCFDJJ34", "RCONJJ34")
concept_panel["held_to_maturity_securities"] = htm_new
concept_panel["held_to_maturity_securities__source"] = htm_new_source
legacy_mask = report_year.le(2018)
concept_panel.loc[legacy_mask, "held_to_maturity_securities"] = htm_legacy.loc[legacy_mask]
concept_panel.loc[legacy_mask, "held_to_maturity_securities__source"] = htm_legacy_source.loc[legacy_mask]

domestic_deposits = pd.to_numeric(raw_panel["RCON2200"], errors="coerce")
foreign_deposits = pd.to_numeric(raw_panel["RCFN2200"], errors="coerce")
concept_panel["total_deposits"] = domestic_deposits + foreign_deposits.fillna(0)
concept_panel.loc[domestic_deposits.isna(), "total_deposits"] = np.nan
concept_panel["total_deposits__source"] = "RCON2200 + RCFN2200 (missing foreign-office amount treated as zero)"

nonaccrual_values, nonaccrual_source, nonaccrual_assignments, nonaccrual_audit = construct_historical_formula("nonaccrual_loans")
past_due_values, past_due_source, past_due_assignments, past_due_audit = construct_historical_formula("past_due_90_days")
concept_panel["nonaccrual_loans"] = nonaccrual_values
concept_panel["nonaccrual_loans__source"] = nonaccrual_source
concept_panel["past_due_90_days"] = past_due_values
concept_panel["past_due_90_days__source"] = past_due_source
formula_application_audit = pd.DataFrame(nonaccrual_audit + past_due_audit)

riad_ytd_mapping = {
    "net_income_ytd": "RIAD4340",
    "net_interest_income_ytd": "RIAD4074",
    "total_interest_expense_ytd": "RIAD4073",
    "total_interest_income_ytd": "RIAD4107",
    "total_noninterest_expense_ytd": "RIAD4093",
    "total_noninterest_income_ytd": "RIAD4079",
}
for output_column, raw_code in riad_ytd_mapping.items():
    concept_panel[output_column] = pd.to_numeric(raw_panel[raw_code], errors="coerce")
    concept_panel[f"{output_column}__source"] = raw_code

concept_panel["provision_credit_losses_ytd"] = pd.to_numeric(raw_panel["RIADJJ33"], errors="coerce")
concept_panel["provision_credit_losses_ytd__source"] = "RIADJJ33"
provision_legacy_mask = report_year.le(2018)
concept_panel.loc[provision_legacy_mask, "provision_credit_losses_ytd"] = raw_panel.loc[provision_legacy_mask, "RIAD4230"]
concept_panel.loc[provision_legacy_mask, "provision_credit_losses_ytd__source"] = "RIAD4230"

ytd_concepts = {
    "net_income", "net_interest_income", "provision_credit_losses",
    "total_interest_expense", "total_interest_income",
    "total_noninterest_expense", "total_noninterest_income",
}
concept_manifest = final_source_mapping[["concept", "measurement_basis", "unit", "mapping_method", "sign_policy"]].copy()
concept_manifest["output_column"] = concept_manifest["concept"].map(
    lambda concept: concept + "_ytd" if concept in ytd_concepts else concept
)
concept_columns = concept_manifest["output_column"].tolist()
missing_concept_columns = [column for column in concept_columns if column not in concept_panel]
if missing_concept_columns:
    raise AssertionError(f"Concept columns not constructed: {missing_concept_columns}")
for column in concept_columns:
    concept_panel[column] = pd.to_numeric(concept_panel[column], errors="coerce")

assert nonaccrual_assignments.eq(1).all()
assert past_due_assignments.eq(1).all()

print(f"Twenty harmonised concepts constructed for {len(concept_panel):,} raw rows.")
print("RIAD flows remain explicitly labelled as YTD.")


## 9. Validate units and signs; exclude irrecoverable total-assets rows


In [ ]:
# Validate units and signs; exclude irrecoverable total-assets rows

unit_validation = concept_manifest[["concept", "output_column", "measurement_basis", "unit"]].copy()
unit_validation["raw_scale_action"] = "No rescaling; retain FFIEC USD-thousands convention"
unit_validation["unit_status"] = "CONFIRMED"

invalid_total_assets_mask = concept_panel["total_assets"].isna() | concept_panel["total_assets"].le(0)
invalid_total_assets_exclusions = concept_panel.loc[
    invalid_total_assets_mask,
    ["report_date", "idrssd", "cert", "bank_name", "filing_form", "total_assets", "total_assets__source"],
].copy()
invalid_total_assets_exclusions["exclusion_reason"] = np.select(
    [
        invalid_total_assets_exclusions["total_assets"].isna(),
        invalid_total_assets_exclusions["total_assets"].eq(0),
        invalid_total_assets_exclusions["total_assets"].lt(0),
    ],
    ["missing_total_assets", "zero_total_assets", "negative_total_assets"],
    default="invalid_total_assets",
)
harmonised_panel = concept_panel.loc[~invalid_total_assets_mask].copy().reset_index(drop=True)

expected_nonnegative_concepts = [
    "allowance_credit_losses", "available_for_sale_securities", "fed_funds_sold",
    "held_to_maturity_securities", "nonaccrual_loans", "other_real_estate_owned",
    "past_due_90_days", "total_assets", "total_deposits", "total_loans", "trading_assets",
]
sign_audit_records = []
for column in concept_columns:
    values = harmonised_panel[column]
    expected_nonnegative = column in expected_nonnegative_concepts
    sign_audit_records.append(
        {
            "output_column": column,
            "observations": len(values),
            "nonmissing": int(values.notna().sum()),
            "negative_values": int(values.lt(0).sum()),
            "zero_values": int(values.eq(0).sum()),
            "positive_values": int(values.gt(0).sum()),
            "expected_nonnegative": expected_nonnegative,
            "sign_review_flag": bool(expected_nonnegative and values.lt(0).any()),
            "treatment": "Values retained exactly; no clipping or absolute-value transformation",
        }
    )
sign_validation = pd.DataFrame(sign_audit_records)

concept_missingness_records = []
for row in concept_manifest.itertuples(index=False):
    values = harmonised_panel[row.output_column]
    concept_missingness_records.append(
        {
            "concept": row.concept,
            "output_column": row.output_column,
            "measurement_basis": row.measurement_basis,
            "observations": len(values),
            "nonmissing_values": int(values.notna().sum()),
            "missing_values": int(values.isna().sum()),
            "nonmissing_rate": float(values.notna().mean()),
            "minimum": values.min(),
            "maximum": values.max(),
        }
    )
concept_missingness = pd.DataFrame(concept_missingness_records).sort_values(
    ["nonmissing_rate", "concept"]
).reset_index(drop=True)

numeric_matrix = harmonised_panel[concept_columns].to_numpy(dtype="float64")
infinite_values = int(np.isinf(numeric_matrix).sum())

print(f"Irrecoverable total-assets rows excluded: {len(invalid_total_assets_exclusions):,}")
print(f"Remaining harmonised rows: {len(harmonised_panel):,}")
print(f"Expected-nonnegative concept sign flags: {sign_validation['sign_review_flag'].sum():,}")
print(f"Infinite concept values: {infinite_values:,}")
display(sign_validation)
display(concept_missingness)


## 10. Run key, date, coverage and institutional-information audits


In [ ]:
# Run key, date, coverage and institutional-information audits

harmonised_panel["report_date"] = pd.to_datetime(harmonised_panel["report_date"]).dt.normalize()
harmonised_panel["idrssd"] = pd.to_numeric(harmonised_panel["idrssd"], errors="coerce").astype("Int64")
harmonised_panel["cert"] = pd.to_numeric(harmonised_panel["cert"], errors="coerce").astype("Int64")

duplicate_idrssd_mask = harmonised_panel.duplicated(["idrssd", "report_date"], keep=False)
duplicate_cert_mask = (
    harmonised_panel["cert"].notna()
    & harmonised_panel.loc[harmonised_panel["cert"].notna()].duplicated(
        ["cert", "report_date"], keep=False
    ).reindex(harmonised_panel.index, fill_value=False)
)
duplicate_idrssd_quarter = harmonised_panel.loc[
    duplicate_idrssd_mask,
    ["report_date", "idrssd", "cert", "bank_name", "filing_form"],
].copy()
duplicate_cert_quarter = harmonised_panel.loc[
    duplicate_cert_mask,
    ["report_date", "idrssd", "cert", "bank_name", "filing_form"],
].copy()

expected_quarters = pd.period_range(
    DATA_SCOPE["expected_ffiec_start"], DATA_SCOPE["expected_ffiec_end"], freq="Q"
)
detected_quarters = pd.PeriodIndex(harmonised_panel["report_date"], freq="Q").unique().sort_values()
missing_panel_quarters = sorted(
    set(expected_quarters.astype(str)) - set(detected_quarters.astype(str)),
    key=lambda value: pd.Period(value, freq="Q"),
)
unexpected_panel_quarters = sorted(
    set(detected_quarters.astype(str)) - set(expected_quarters.astype(str)),
    key=lambda value: pd.Period(value, freq="Q"),
)

annual_panel_summary = (
    harmonised_panel.assign(year=harmonised_panel["report_date"].dt.year)
    .groupby("year", as_index=False)
    .agg(
        observations=("idrssd", "size"),
        unique_idrssd=("idrssd", "nunique"),
        unique_cert=("cert", "nunique"),
        quarters=("report_date", "nunique"),
        first_report_date=("report_date", "min"),
        last_report_date=("report_date", "max"),
    )
)
institutional_information_audit = pd.DataFrame(
    [
        {"field": "idrssd", "missing": int(harmonised_panel["idrssd"].isna().sum()), "nonmissing_rate": float(harmonised_panel["idrssd"].notna().mean()), "required": True},
        {"field": "cert", "missing": int(harmonised_panel["cert"].isna().sum()), "nonmissing_rate": float(harmonised_panel["cert"].notna().mean()), "required": False},
        {"field": "bank_name", "missing": int(harmonised_panel["bank_name"].isna().sum()), "nonmissing_rate": float(harmonised_panel["bank_name"].notna().mean()), "required": False},
        {"field": "bank_state", "missing": int(harmonised_panel["bank_state"].isna().sum()), "nonmissing_rate": float(harmonised_panel["bank_state"].notna().mean()), "required": False},
        {"field": "filing_form", "missing": int(harmonised_panel["filing_form"].isna().sum()), "nonmissing_rate": float(harmonised_panel["filing_form"].notna().mean()), "required": True},
    ]
)

print(f"Duplicate IDRSSD-quarter rows: {len(duplicate_idrssd_quarter):,}")
print(f"Duplicate CERT-quarter rows: {len(duplicate_cert_quarter):,}")
print(f"Reporting quarters: {detected_quarters[0]} to {detected_quarters[-1]} ({len(detected_quarters)})")
print(f"Missing panel quarters: {missing_panel_quarters}")
display(institutional_information_audit)
display(annual_panel_summary)


## 11. Save audit outputs, enforce the gate and write the harmonised V2 panel


In [ ]:
# Save audit outputs, enforce the gate and write the harmonised V2 panel

current_mapping_sha256 = dataframe_sha256(final_source_mapping)
current_formula_sha256 = dataframe_sha256(final_historical_formula_registry)
mapping_hash_matches = current_mapping_sha256 == mapping_lock_manifest["source_mapping_sha256"]
formula_hash_matches = current_formula_sha256 == mapping_lock_manifest["formula_registry_sha256"]

extraction_checks = {
    "Notebook 02 prerequisite passed": nb02_audit.get("overall_status") == "PASS",
    "Locked source mapping fingerprint matches": mapping_hash_matches,
    "Locked formula-registry fingerprint matches": formula_hash_matches,
    "All locked code windows passed": incomplete_code_windows.empty,
    "Raw files cover 2001-2026": len(missing_file_years) == 0,
    "No invalid report dates": int(file_read_audit["invalid_report_date_rows"].sum()) == 0,
    "All raw report dates were quarter ends": int(file_read_audit["non_quarter_end_raw_rows"].sum()) == 0,
    "No invalid IDRSSD values": harmonised_panel["idrssd"].isna().sum() == 0,
    "All filing forms are supported": unsupported_form_rows == 0,
    "No conflicting values across file parts": merge_value_conflicts == 0,
    "No numeric conversion failures": numeric_conversion_failures == 0,
    "Every delinquency row has exactly one formula": nonaccrual_assignments.eq(1).all() and past_due_assignments.eq(1).all(),
    "All 20 concepts exist": len(concept_columns) == 20 and all(column in harmonised_panel for column in concept_columns),
    "No infinite concept values": infinite_values == 0,
    "Total assets are positive after documented exclusions": harmonised_panel["total_assets"].gt(0).all(),
    "No duplicate IDRSSD-quarter rows": duplicate_idrssd_quarter.empty,
    "No duplicate CERT-quarter rows": duplicate_cert_quarter.empty,
    "Complete expected quarter coverage": not missing_panel_quarters and not unexpected_panel_quarters,
    "YTD flows remain explicitly labelled": all(
        concept_manifest.loc[concept_manifest["measurement_basis"].eq("year_to_date_flow"), "output_column"].str.endswith("_ytd")
    ),
}
extraction_gate = pd.DataFrame(
    {"check": list(extraction_checks), "passed": list(extraction_checks.values())}
)
extraction_gate["status"] = extraction_gate["passed"].map({True: "PASS", False: "FAIL"})

output_tables = {
    "locked_source_mapping_v2.csv": final_source_mapping,
    "locked_historical_formula_registry_v2.csv": final_historical_formula_registry,
    "locked_code_coverage_validation_v2.csv": locked_code_coverage_validation,
    "raw_extraction_file_inventory_v2.csv": raw_extraction_file_inventory,
    "file_read_audit_v2.csv": file_read_audit,
    "part_merge_conflicts_v2.csv": part_merge_conflicts,
    "concept_manifest_v2.csv": concept_manifest,
    "formula_application_audit_v2.csv": formula_application_audit,
    "unit_validation_v2.csv": unit_validation,
    "sign_validation_v2.csv": sign_validation,
    "concept_missingness_v2.csv": concept_missingness,
    "invalid_total_assets_exclusions_v2.csv": invalid_total_assets_exclusions,
    "duplicate_idrssd_quarter_v2.csv": duplicate_idrssd_quarter,
    "duplicate_cert_quarter_v2.csv": duplicate_cert_quarter,
    "institutional_information_audit_v2.csv": institutional_information_audit,
    "annual_panel_summary_v2.csv": annual_panel_summary,
    "extraction_gate_v2.csv": extraction_gate,
}
for filename, table in output_tables.items():
    output_path = NB03_OUTPUT_DIR / filename
    assert output_path.is_relative_to(V2_ROOT)
    table.to_csv(output_path, index=False)

lock_manifest_path = NB03_OUTPUT_DIR / "mapping_lock_manifest_v2.json"
lock_manifest_path.write_text(json.dumps(mapping_lock_manifest, indent=2) + "\n", encoding="utf-8")

failed_checks = extraction_gate.loc[~extraction_gate["passed"], "check"].tolist()
display(extraction_gate)
if failed_checks:
    raise AssertionError("Notebook 03 extraction gate failed: " + "; ".join(failed_checks))

harmonised_panel = harmonised_panel.sort_values(["report_date", "idrssd"]).reset_index(drop=True)
temp_panel_path = HARMONISED_PANEL_PATH.with_suffix(".tmp.parquet")
harmonised_panel.to_parquet(temp_panel_path, index=False)
os.replace(temp_panel_path, HARMONISED_PANEL_PATH)

import pyarrow.parquet as pq
parquet_metadata = pq.ParquetFile(HARMONISED_PANEL_PATH).metadata
assert parquet_metadata.num_rows == len(harmonised_panel)
assert parquet_metadata.num_columns == len(harmonised_panel.columns)

warning_messages = []
if not invalid_total_assets_exclusions.empty:
    warning_messages.append(f"{len(invalid_total_assets_exclusions)} invalid total-assets rows excluded")
sign_review_count = int(sign_validation["sign_review_flag"].sum())
if sign_review_count:
    warning_messages.append(f"{sign_review_count} expected-nonnegative concepts contain negative values")

overall_status = "PASS"
audit_report = {
    "notebook": "03_FFIEC_Data_Extraction_and_Harmonisation_V2",
    "analysis_version": VERSION_2_MAIN_TIMING_POLICY["analysis_version"],
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "overall_status": overall_status,
    "warnings": warning_messages,
    "raw_rows_before_asset_exclusions": int(len(concept_panel)),
    "invalid_total_assets_rows_excluded": int(len(invalid_total_assets_exclusions)),
    "harmonised_rows": int(len(harmonised_panel)),
    "unique_idrssd": int(harmonised_panel["idrssd"].nunique()),
    "unique_cert": int(harmonised_panel["cert"].nunique()),
    "reporting_quarters": int(len(detected_quarters)),
    "first_report_date": str(harmonised_panel["report_date"].min().date()),
    "last_report_date": str(harmonised_panel["report_date"].max().date()),
    "concept_count": int(len(concept_columns)),
    "duplicate_idrssd_quarter_rows": int(len(duplicate_idrssd_quarter)),
    "duplicate_cert_quarter_rows": int(len(duplicate_cert_quarter)),
    "numeric_conversion_failures": numeric_conversion_failures,
    "merge_value_conflicts": merge_value_conflicts,
    "mapping_sha256": mapping_sha256,
    "formula_registry_sha256": formula_registry_sha256,
    "panel_path": str(HARMONISED_PANEL_PATH),
    "output_folder": str(NB03_OUTPUT_DIR),
}
audit_report_path = NB03_OUTPUT_DIR / "audit_report_v2.json"
audit_report_path.write_text(json.dumps(audit_report, indent=2) + "\n", encoding="utf-8")

print(f"Harmonised V2 panel saved and verified:\n{HARMONISED_PANEL_PATH}")
print(f"Rows: {len(harmonised_panel):,}; columns: {len(harmonised_panel.columns):,}")


## 12. Final result


In [ ]:
# Final result

print("=" * 78)
print("NOTEBOOK 03 V2 — FINAL HARMONISATION RESULT")
print("=" * 78)
print(f"Overall status: {overall_status}")
print(f"Raw rows before total-assets exclusions: {len(concept_panel):,}")
print(f"Invalid total-assets rows excluded: {len(invalid_total_assets_exclusions):,}")
print(f"Harmonised rows: {len(harmonised_panel):,}")
print(f"Unique IDRSSD: {harmonised_panel['idrssd'].nunique():,}")
print(f"Unique CERT: {harmonised_panel['cert'].nunique():,}")
print(f"Reporting quarters: {len(detected_quarters):,}")
print(f"Reporting period: {harmonised_panel['report_date'].min().date()} to {harmonised_panel['report_date'].max().date()}")
print(f"Harmonised concepts: {len(concept_columns):,}")
print(f"Duplicate IDRSSD-quarter rows: {len(duplicate_idrssd_quarter):,}")
print(f"Duplicate CERT-quarter rows: {len(duplicate_cert_quarter):,}")
print(f"Numeric conversion failures: {numeric_conversion_failures:,}")
print(f"Merge-value conflicts: {merge_value_conflicts:,}")
print(f"Expected-nonnegative concept sign flags: {sign_review_count:,}")
print(f"Panel: {HARMONISED_PANEL_PATH}")
print(f"Audit folder: {NB03_OUTPUT_DIR}")

assert audit_report_path.exists()
assert HARMONISED_PANEL_PATH.exists()
assert extraction_gate["passed"].all()
if warning_messages:
    print("\nNotebook 03 completed with warnings:")
    for warning in warning_messages:
        print(f"- {warning}")
    print("Review the final results and relevant warning audit rows before Notebook 04.")
else:
    print("\nNotebook 03 completed successfully with no warnings.")
print("Do not run Notebook 04 until this harmonisation audit has been reviewed.")
